# Adjoint Differentiation Benchmark

**Superfermion (Rust) vs PennyLane Lightning (C++)**

Compares gradient computation speed using the adjoint differentiation method.
Both implementations use the same algorithm (Jones & Gacon 2020) but in
different compiled backends.

### Methodology
- Hardware-efficient ansatz (RY-RZ-CNOT layers) at varying qubit counts and depths
- Observable: full Pauli Hamiltonian (ZZ chain + X transverse field)
- Per-size warmup + 15 trials, report median and IQR
- Correctness: cross-check gradients between SF, PennyLane, and finite differences
- VQE H2 end-to-end: both frameworks find H2 ground state energy

In [ ]:
import numpy as np
import time
import gc
from statistics import median

import superfermion as sf
from superfermion.qml.gradient.adjoint import adjoint_grad_vector
from superfermion.observables.core import Hamiltonian
from superfermion.chemistry.hamiltonians import PauliString

import pennylane as qml
from pennylane import numpy as pnp

N_TRIALS = 15

def timed(fn):
    gc.disable()
    t0 = time.perf_counter()
    result = fn()
    elapsed = (time.perf_counter() - t0) * 1000
    gc.enable()
    return result, elapsed

def benchmark(fn, n_trials=N_TRIALS):
    times = []
    for _ in range(n_trials):
        _, t = timed(fn)
        times.append(round(t, 2))
    med = median(times)
    s = sorted(times)
    q1 = s[len(s)//4]
    q3 = s[3*len(s)//4]
    return med, times, q1, q3

def fmt(sf_t, other_t):
    if sf_t < other_t:
        return f'SF {other_t/sf_t:.1f}x faster'
    else:
        return f'PL {sf_t/other_t:.1f}x faster'

print(f'PennyLane {qml.__version__}')
print(f'Lightning: {qml.device("lightning.qubit", wires=1).name}')
print(f'Trials: {N_TRIALS}')

## 1. Correctness: Cross-check Gradients

In [ ]:
n_qubits = 4
n_params = 8  # 2 params per qubit (RY + RZ)

# Build TFIM Hamiltonian: H = -sum ZZ - 0.5 sum X
def build_tfim_hamiltonian(n):
    terms = []
    for i in range(n - 1):
        ps = ['I'] * n
        ps[i] = 'Z'
        ps[i+1] = 'Z'
        terms.append(PauliString(''.join(ps), -1.0))
    for i in range(n):
        ps = ['I'] * n
        ps[i] = 'X'
        terms.append(PauliString(''.join(ps), -0.5))
    return Hamiltonian(terms)

H = build_tfim_hamiltonian(n_qubits)
print(f'TFIM Hamiltonian: {len(H.terms)} terms on {n_qubits} qubits')

# SF circuit
param_names = [f'p{i}' for i in range(n_params)]
sf_params = [sf.param(name) for name in param_names]
c = sf.Circuit(n_qubits)
for i in range(n_qubits):
    c.ry(sf_params[2*i], i)
    c.rz(sf_params[2*i+1], i)
for i in range(n_qubits - 1):
    c.cx(i, i+1)

np.random.seed(42)
vals = np.random.uniform(-np.pi, np.pi, n_params)

# SF adjoint gradient (Rust)
sf_grad = adjoint_grad_vector(c, H, param_names, vals)

# PennyLane Lightning adjoint gradient
dev = qml.device('lightning.qubit', wires=n_qubits)

H_pl = qml.Hamiltonian(
    [t.coeffs for t in H.terms],
    [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
     if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
     for t in H.terms]
)

@qml.qnode(dev, diff_method='adjoint')
def pl_circuit(params):
    for i in range(n_qubits):
        qml.RY(params[2*i], wires=i)
        qml.RZ(params[2*i+1], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])
    return qml.expval(H_pl)

pl_params = pnp.array(vals, requires_grad=True)
pl_grad = qml.grad(pl_circuit)(pl_params)

# Finite differences (ground truth)
eps = 1e-5
fd_grad = np.zeros(n_params)
for i in range(n_params):
    v_plus = vals.copy(); v_plus[i] += eps
    v_minus = vals.copy(); v_minus[i] -= eps
    sv_plus = sf.run(c.bind(dict(zip(param_names, v_plus))), device='cpu', shots=0).statevector
    sv_minus = sf.run(c.bind(dict(zip(param_names, v_minus))), device='cpu', shots=0).statevector
    e_plus = float(np.real(H._fast_expval(sv_plus)))
    e_minus = float(np.real(H._fast_expval(sv_minus)))
    fd_grad[i] = (e_plus - e_minus) / (2 * eps)

print(f'\n{"param":>6s} {"SF adj":>10s} {"PL adj":>10s} {"FD":>10s}')
print('-' * 42)
for i in range(n_params):
    print(f'{param_names[i]:>6s} {sf_grad[i]:+10.6f} {pl_grad[i]:+10.6f} {fd_grad[i]:+10.6f}')

sf_pl_diff = np.max(np.abs(sf_grad - pl_grad))
sf_fd_diff = np.max(np.abs(sf_grad - fd_grad))
print(f'\nMax |SF - PL|: {sf_pl_diff:.2e}')
print(f'Max |SF - FD|: {sf_fd_diff:.2e}')
assert sf_fd_diff < 1e-4, f'SF vs FD mismatch: {sf_fd_diff}'

## 2. Gradient Speed: SF Adjoint vs PennyLane Lightning Adjoint

In [ ]:
# Scale up: vary qubit count with 2 params/qubit, depth=1
qubit_sizes = [4, 6, 8, 10, 12, 14, 16, 18]
grad_results = []

print('='*80)
print('ADJOINT GRADIENT BENCHMARK (HE ansatz, depth=1, TFIM observable)')
print('='*80)

# Global warmup
print(f'Warming up at n={qubit_sizes[-1]}...')
n_warm = qubit_sizes[-1]
npar_warm = 2 * n_warm
H_warm = build_tfim_hamiltonian(n_warm)
pn_warm = [f'p{i}' for i in range(npar_warm)]
sp_warm = [sf.param(nm) for nm in pn_warm]
c_warm = sf.Circuit(n_warm)
for i in range(n_warm):
    c_warm.ry(sp_warm[2*i], i)
    c_warm.rz(sp_warm[2*i+1], i)
for i in range(n_warm - 1):
    c_warm.cx(i, i+1)
v_warm = np.random.uniform(-np.pi, np.pi, npar_warm)
adjoint_grad_vector(c_warm, H_warm, pn_warm, v_warm)

dev_warm = qml.device('lightning.qubit', wires=n_warm)
H_pl_warm = qml.Hamiltonian(
    [t.coeffs for t in H_warm.terms],
    [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
     if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
     for t in H_warm.terms]
)

@qml.qnode(dev_warm, diff_method='adjoint')
def pl_warm_fn(params):
    for i in range(n_warm):
        qml.RY(params[2*i], wires=i)
        qml.RZ(params[2*i+1], wires=i)
    for i in range(n_warm - 1):
        qml.CNOT(wires=[i, i+1])
    return qml.expval(H_pl_warm)

pl_p_warm = pnp.array(v_warm, requires_grad=True)
qml.grad(pl_warm_fn)(pl_p_warm)
print('Done.\n')

for n in qubit_sizes:
    n_par = 2 * n
    H_n = build_tfim_hamiltonian(n)
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n)
    for i in range(n):
        circ.ry(sparams[2*i], i)
        circ.rz(sparams[2*i+1], i)
    for i in range(n - 1):
        circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    # PennyLane setup
    dev_n = qml.device('lightning.qubit', wires=n)
    H_pl_n = qml.Hamiltonian(
        [t.coeffs for t in H_n.terms],
        [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
         if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
         for t in H_n.terms]
    )

    @qml.qnode(dev_n, diff_method='adjoint')
    def pl_fn(params):
        for i in range(n):
            qml.RY(params[2*i], wires=i)
            qml.RZ(params[2*i+1], wires=i)
        for i in range(n - 1):
            qml.CNOT(wires=[i, i+1])
        return qml.expval(H_pl_n)

    pl_p = pnp.array(vv, requires_grad=True)

    # Per-size warmup
    adjoint_grad_vector(circ, H_n, pnames, vv)
    qml.grad(pl_fn)(pl_p)

    # Benchmark SF
    def run_sf():
        return adjoint_grad_vector(circ, H_n, pnames, vv)

    sf_med, sf_times, sf_q1, sf_q3 = benchmark(run_sf)

    # Benchmark PL
    def run_pl():
        return qml.grad(pl_fn)(pl_p)

    pl_med, pl_times, pl_q1, pl_q3 = benchmark(run_pl)

    grad_results.append({
        'n': n, 'n_params': n_par,
        'sf_med': sf_med, 'sf_q1': sf_q1, 'sf_q3': sf_q3,
        'pl_med': pl_med, 'pl_q1': pl_q1, 'pl_q3': pl_q3,
    })

    print(f'n={n:2d} ({n_par:2d} params) | '
          f'SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'PL {pl_med:8.1f}ms [{pl_q1:.1f}-{pl_q3:.1f}] | '
          f'{fmt(sf_med, pl_med)}')

## 3. Scaling with Depth (fixed n=10)

In [ ]:
n_fixed = 10
depths = [1, 2, 3, 5, 8, 12]
depth_results = []

print('='*80)
print(f'DEPTH SCALING (n={n_fixed}, TFIM, adjoint gradient)')
print('='*80)

H_fixed = build_tfim_hamiltonian(n_fixed)

for depth in depths:
    n_par = 2 * n_fixed * depth
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n_fixed)
    pidx = 0
    for _ in range(depth):
        for i in range(n_fixed):
            circ.ry(sparams[pidx], i); pidx += 1
            circ.rz(sparams[pidx], i); pidx += 1
        for i in range(n_fixed - 1):
            circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    # PennyLane
    dev_d = qml.device('lightning.qubit', wires=n_fixed)
    H_pl_d = qml.Hamiltonian(
        [t.coeffs for t in H_fixed.terms],
        [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
         if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
         for t in H_fixed.terms]
    )

    @qml.qnode(dev_d, diff_method='adjoint')
    def pl_depth_fn(params):
        pidx = 0
        for _ in range(depth):
            for i in range(n_fixed):
                qml.RY(params[pidx], wires=i); pidx += 1
                qml.RZ(params[pidx], wires=i); pidx += 1
            for i in range(n_fixed - 1):
                qml.CNOT(wires=[i, i+1])
        return qml.expval(H_pl_d)

    pl_p = pnp.array(vv, requires_grad=True)

    # Per-size warmup
    adjoint_grad_vector(circ, H_fixed, pnames, vv)
    qml.grad(pl_depth_fn)(pl_p)

    def run_sf():
        return adjoint_grad_vector(circ, H_fixed, pnames, vv)

    sf_med, _, sf_q1, sf_q3 = benchmark(run_sf)

    def run_pl():
        return qml.grad(pl_depth_fn)(pl_p)

    pl_med, _, pl_q1, pl_q3 = benchmark(run_pl)

    depth_results.append({
        'd': depth, 'n_params': n_par,
        'sf_med': sf_med, 'pl_med': pl_med,
    })

    print(f'd={depth:2d} ({n_par:3d} params) | '
          f'SF {sf_med:8.1f}ms [{sf_q1:.1f}-{sf_q3:.1f}] | '
          f'PL {pl_med:8.1f}ms [{pl_q1:.1f}-{pl_q3:.1f}] | '
          f'{fmt(sf_med, pl_med)}')

## 4. VQE H2 End-to-End

In [ ]:
print('='*80)
print('VQE H2 GROUND STATE (STO-3G, 2 qubits, adjoint gradient)')
print('='*80)

# === Superfermion VQE ===
from superfermion.chemistry.hamiltonians import get_molecular_hamiltonian
from superfermion.chemistry.ansatz import uccsd_ansatz
from superfermion.algorithms.variational import VQE

H_h2 = get_molecular_hamiltonian('H2')
ansatz_sf = uccsd_ansatz(2, 2)

t0 = time.perf_counter()
vqe_sf = VQE(ansatz_sf, H_h2, diff_method='adjoint')
result_sf = vqe_sf.minimize(tol=1e-10)
t_sf = (time.perf_counter() - t0) * 1000

print(f'SF VQE: E = {result_sf.optimal_value:.10f} Ha  ({t_sf:.0f} ms, {result_sf.metadata["n_fun_evals"]} evals)')

# === PennyLane Lightning VQE ===
dev_h2 = qml.device('lightning.qubit', wires=2)

H_pl_h2 = qml.Hamiltonian(
    [t.coeffs for t in H_h2.terms],
    [qml.prod(*[getattr(qml, f'Pauli{ch}')(w) for w, ch in enumerate(t.pauli_str) if ch != 'I'])
     if any(ch != 'I' for ch in t.pauli_str) else qml.Identity(0)
     for t in H_h2.terms]
)

@qml.qnode(dev_h2, diff_method='adjoint')
def pl_h2_circuit(theta):
    qml.PauliX(wires=0)
    qml.RY(theta, wires=1)
    qml.CNOT(wires=[1, 0])
    return qml.expval(H_pl_h2)

from scipy.optimize import minimize as scipy_minimize
t0 = time.perf_counter()
pl_result = scipy_minimize(
    lambda x: float(pl_h2_circuit(pnp.array(x[0], requires_grad=True))),
    x0=[0.5],
    jac=lambda x: [float(qml.grad(pl_h2_circuit)(pnp.array(x[0], requires_grad=True)))],
    method='L-BFGS-B',
    tol=1e-10,
)
t_pl = (time.perf_counter() - t0) * 1000

print(f'PL VQE: E = {pl_result.fun:.10f} Ha  ({t_pl:.0f} ms, {pl_result.nfev} evals)')
print(f'\nKnown exact: -1.1373060401 Ha')
print(f'SF error:    {abs(result_sf.optimal_value - (-1.13730604)):.2e} Ha')
print(f'PL error:    {abs(pl_result.fun - (-1.13730604)):.2e} Ha')
print(f'Both within chemical accuracy (1.6 mHa): '
      f'{abs(result_sf.optimal_value - (-1.13730604)) < 0.0016 and abs(pl_result.fun - (-1.13730604)) < 0.0016}')

## 5. Adjoint vs Parameter-Shift Speedup (SF only)

In [ ]:
from superfermion.qml.gradient.parameter_shift import parameter_shift_grad_vector

print('='*80)
print('ADJOINT vs PARAMETER-SHIFT (SF, n=10, TFIM)')
print('='*80)

n_test = 10
H_test = build_tfim_hamiltonian(n_test)
method_results = []

for depth in [1, 2, 4, 8]:
    n_par = 2 * n_test * depth
    pnames = [f'p{i}' for i in range(n_par)]
    sparams = [sf.param(nm) for nm in pnames]
    circ = sf.Circuit(n_test)
    pidx = 0
    for _ in range(depth):
        for i in range(n_test):
            circ.ry(sparams[pidx], i); pidx += 1
            circ.rz(sparams[pidx], i); pidx += 1
        for i in range(n_test - 1):
            circ.cx(i, i+1)
    vv = np.random.uniform(-np.pi, np.pi, n_par)

    # Warmup
    adjoint_grad_vector(circ, H_test, pnames, vv)
    parameter_shift_grad_vector(circ, H_test, pnames, vv)

    def run_adj():
        return adjoint_grad_vector(circ, H_test, pnames, vv)

    def run_ps():
        return parameter_shift_grad_vector(circ, H_test, pnames, vv)

    adj_med, _, _, _ = benchmark(run_adj)
    ps_med, _, _, _ = benchmark(run_ps)

    method_results.append({'depth': depth, 'n_params': n_par, 'adj': adj_med, 'ps': ps_med})

    print(f'd={depth:2d} ({n_par:3d} params) | '
          f'Adjoint {adj_med:8.2f}ms | '
          f'Param-shift {ps_med:8.1f}ms | '
          f'Adjoint {ps_med/adj_med:.1f}x faster')

## 6. Summary

In [ ]:
import json

summary = {
    'gradient_scaling': grad_results,
    'depth_scaling': depth_results,
    'adjoint_vs_paramshift': method_results,
}

with open('benchmark_adjoint_data.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Results saved to benchmark_adjoint_data.json')
print()
print('Key findings:')
print(f'  Gradient scaling (n=4..{qubit_sizes[-1]}):')
for r in grad_results:
    ratio = r['pl_med'] / r['sf_med'] if r['sf_med'] < r['pl_med'] else -(r['sf_med'] / r['pl_med'])
    winner = 'SF' if ratio > 0 else 'PL'
    print(f'    n={r["n"]:2d}: SF {r["sf_med"]:.2f}ms vs PL {r["pl_med"]:.1f}ms ({winner} {abs(ratio):.1f}x)')
print(f'  Adjoint vs param-shift (SF, n={n_test}):')
for r in method_results:
    print(f'    d={r["depth"]:2d} ({r["n_params"]:3d}p): adj {r["adj"]:.2f}ms vs ps {r["ps"]:.1f}ms ({r["ps"]/r["adj"]:.1f}x adjoint advantage)')